[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/17_sequence_and_state_space.ipynb)

# 17. Sequence recurrence and state-space updates — from SSM to paper-faithful tiny Mamba

Mamba의 핵심 계산을

continuous SSM → discretization → associative recurrence → selective `Δ/B/C` → causal depthwise convolution → selective scan → gate

순서로 본다.

이번 버전에서는 Mamba block의 중요한 low-rank timestep parameterization도 유지한다.

`x_proj: d_inner -> dt_rank + 2*d_state`

`dt_proj: dt_rank -> d_inner`

즉 `Δ`를 `Linear(d_inner, d_inner)` 하나로 바로 만드는 shortcut을 쓰지 않는다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Linear state-space recurrence

Discrete SSM은 fixed-size state를 갱신한다.

`h_t = A_bar h_(t-1) + B_bar x_t`

`y_t = C h_t + D x_t`


In [ ]:
inputs = torch.tensor(
    [1.0, 0.0, -1.0, 0.5],
    device=device,
)

A_bar = torch.tensor(
    [
        [0.8, 0.1],
        [0.0, 0.9],
    ],
    device=device,
)

B_bar = torch.tensor(
    [1.0, 0.5],
    device=device,
)

C = torch.tensor(
    [0.7, -0.2],
    device=device,
)

D = torch.tensor(
    0.1,
    device=device,
)

state = torch.zeros(
    2,
    device=device,
)

outputs = []

for x_t in inputs:
    state = (
        A_bar @ state
        + B_bar * x_t
    )

    y_t = (
        C @ state
        + D * x_t
    )

    outputs.append(y_t)

print(
    "outputs:",
    torch.stack(outputs),
)


## 2. Continuous SSM to discrete transition

Continuous dynamics

`dh/dt = A h + B x`

를 token별 step size `Δ`로 discretize하면 diagonal `A`에서는 state transition이 `exp(Δ A)`로 나타난다.


In [ ]:
A = torch.tensor(
    [-1.0, -2.0],
    device=device,
)

B = torch.tensor(
    [1.0, 0.5],
    device=device,
)

delta = torch.tensor(
    0.25,
    device=device,
)

A_discrete = torch.exp(
    delta * A
)

B_input = delta * B

print(
    "exp(delta * A):",
    A_discrete,
)
print(
    "delta * B:",
    B_input,
)


## 3. Associative recurrence viewpoint

Affine recurrence `h' = a h + b`는 pair `(a, b)`의 associative composition으로 표현할 수 있다.

실제 Mamba CUDA kernel은 이런 구조를 이용해 sequence scan을 병렬화한다. 여기서는 계산 구조 검증이 목적이므로 읽기 쉬운 sequential reference scan을 사용한다.


In [ ]:
a1 = torch.tensor(
    0.5,
    device=device,
)
b1 = torch.tensor(
    1.0,
    device=device,
)

a2 = torch.tensor(
    0.8,
    device=device,
)
b2 = torch.tensor(
    2.0,
    device=device,
)

a_composed = a2 * a1
b_composed = a2 * b1 + b2

h0 = torch.tensor(
    3.0,
    device=device,
)

sequential = (
    a2 * (
        a1 * h0
        + b1
    )
    + b2
)

composed = (
    a_composed * h0
    + b_composed
)

print(
    "sequential:",
    sequential,
)
print(
    "composed:",
    composed,
)


## 4. Reference selective scan

Mamba의 선택성은 token마다 `Δ_t`, `B_t`, `C_t`가 달라지는 데 있다.

`A`는 learned diagonal state dynamics이고,

- `Δ_t`는 transition time scale
- `B_t`는 input을 state에 쓰는 방식
- `C_t`는 state를 읽는 방식

을 input-dependent하게 만든다.


In [ ]:
def selective_scan_reference(
    u,
    delta,
    A,
    B,
    C,
    D,
):
    batch_size = u.size(0)
    channels = u.size(1)
    sequence_length = u.size(2)
    state_dim = A.size(1)

    state = torch.zeros(
        batch_size,
        channels,
        state_dim,
        device=u.device,
        dtype=u.dtype,
    )

    outputs = []

    for time_index in range(
        sequence_length
    ):
        u_t = u[
            :,
            :,
            time_index,
        ]

        delta_t = delta[
            :,
            :,
            time_index,
        ]

        B_t = B[
            :,
            time_index,
            :,
        ]

        C_t = C[
            :,
            time_index,
            :,
        ]

        dA = torch.exp(
            delta_t[:, :, None]
            * A[None, :, :]
        )

        dB = (
            delta_t[:, :, None]
            * B_t[:, None, :]
        )

        state = (
            dA * state
            + dB
            * u_t[:, :, None]
        )

        y_t = torch.sum(
            state
            * C_t[:, None, :],
            dim=-1,
        )

        y_t = (
            y_t
            + D[None, :] * u_t
        )

        outputs.append(y_t)

    return torch.stack(
        outputs,
        dim=-1,
    )


## 5. Paper-faithful tiny Mamba block

공식 Mamba block의 parameter topology를 작은 dimension으로 유지한다.

1. `in_proj`가 `x` branch와 `z` gate branch를 만든다.
2. `x` branch에 causal depthwise convolution을 적용한다.
3. `x_proj`가 한 번에 low-rank `dt`, `B`, `C`를 만든다.
4. `dt_proj`가 `dt_rank -> d_inner`로 expand한다.
5. `softplus(dt + dt_bias)`로 positive timestep을 만든다.
6. selective scan을 수행한다.
7. scan output에 `SiLU(z)` gate를 곱하고 `out_proj`를 적용한다.

CUDA fused kernel은 생략하지만 block의 학습 parameterization은 바꾸지 않는다.


In [ ]:
class TinyMambaBlock(nn.Module):
    def __init__(
        self,
        model_dim=8,
        state_dim=4,
        conv_kernel=3,
        expand=2,
        dt_rank=2,
        dt_min=0.001,
        dt_max=0.1,
    ):
        super().__init__()

        self.model_dim = model_dim
        self.state_dim = state_dim
        self.conv_kernel = conv_kernel
        self.inner_dim = (
            expand * model_dim
        )
        self.dt_rank = dt_rank

        self.in_proj = nn.Linear(
            model_dim,
            2 * self.inner_dim,
            bias=False,
        )

        self.conv1d = nn.Conv1d(
            self.inner_dim,
            self.inner_dim,
            kernel_size=conv_kernel,
            groups=self.inner_dim,
            padding=0,
            bias=True,
        )

        self.x_proj = nn.Linear(
            self.inner_dim,
            dt_rank
            + 2 * state_dim,
            bias=False,
        )

        self.dt_proj = nn.Linear(
            dt_rank,
            self.inner_dim,
            bias=True,
        )

        dt_init_std = (
            dt_rank ** -0.5
        )

        nn.init.uniform_(
            self.dt_proj.weight,
            -dt_init_std,
            dt_init_std,
        )

        initial_dt = torch.exp(
            torch.rand(
                self.inner_dim
            )
            * (
                math.log(dt_max)
                - math.log(dt_min)
            )
            + math.log(dt_min)
        )

        inverse_softplus = (
            initial_dt
            + torch.log(
                -torch.expm1(
                    -initial_dt
                )
            )
        )

        with torch.no_grad():
            self.dt_proj.bias.copy_(
                inverse_softplus
            )

        initial_A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32,
        )

        initial_A = initial_A.repeat(
            self.inner_dim,
            1,
        )

        self.A_log = nn.Parameter(
            torch.log(initial_A)
        )

        self.D = nn.Parameter(
            torch.ones(
                self.inner_dim
            )
        )

        self.out_proj = nn.Linear(
            self.inner_dim,
            model_dim,
            bias=False,
        )

    def forward(self, hidden):
        projected = self.in_proj(
            hidden
        )

        x_branch, z_branch = (
            projected.chunk(
                2,
                dim=-1,
            )
        )

        x_branch = x_branch.transpose(
            1,
            2,
        )

        left_padding = (
            self.conv_kernel - 1
        )

        x_branch = F.pad(
            x_branch,
            (
                left_padding,
                0,
            ),
        )

        convolved = self.conv1d(
            x_branch
        )
        convolved = F.silu(
            convolved
        )

        sequence_features = (
            convolved.transpose(
                1,
                2,
            )
        )

        projected_ssm = self.x_proj(
            sequence_features
        )

        (
            dt_low_rank,
            B,
            C,
        ) = torch.split(
            projected_ssm,
            [
                self.dt_rank,
                self.state_dim,
                self.state_dim,
            ],
            dim=-1,
        )

        dt = self.dt_proj(
            dt_low_rank
        )
        dt = F.softplus(
            dt
        ).transpose(
            1,
            2,
        )

        A = -torch.exp(
            self.A_log.float()
        )

        scanned = selective_scan_reference(
            convolved,
            dt,
            A,
            B,
            C,
            self.D,
        )

        gated = (
            scanned.transpose(
                1,
                2,
            )
            * F.silu(
                z_branch
            )
        )

        output = self.out_proj(
            gated
        )

        diagnostics = {
            "dt_low_rank": dt_low_rank,
            "dt": dt,
            "B": B,
            "C": C,
            "A": A,
            "convolved": convolved,
        }

        return output, diagnostics


mamba = TinyMambaBlock().to(
    device
)

sequence = torch.randn(
    2,
    6,
    8,
    device=device,
)

mamba_output, diagnostics = mamba(
    sequence
)

print(
    "input:",
    sequence.shape,
)
print(
    "dt low-rank:",
    diagnostics[
        "dt_low_rank"
    ].shape,
)
print(
    "expanded dt:",
    diagnostics["dt"].shape,
)
print(
    "B:",
    diagnostics["B"].shape,
)
print(
    "C:",
    diagnostics["C"].shape,
)
print(
    "output:",
    mamba_output.shape,
)


## 6. Causality sanity check

미래 token만 바꿨을 때 이전 위치의 Mamba output이 바뀌지 않는지 확인한다.

Causal convolution과 recurrent selective scan이 제대로 구현되었다면 prefix output은 동일해야 한다.


In [ ]:
mamba.eval()

original = torch.randn(
    1,
    7,
    8,
    device=device,
)

modified = original.clone()
modified[
    :,
    5:,
    :,
] += 10.0

with torch.no_grad():
    original_output, _ = mamba(
        original
    )
    modified_output, _ = mamba(
        modified
    )

prefix_difference = (
    original_output[
        :,
        :5,
        :,
    ]
    - modified_output[
        :,
        :5,
        :,
    ]
).abs().max()

print(
    "max prefix difference:",
    prefix_difference.item(),
)


## 7. Gradient-flow sanity check

Low-rank timestep path `x_proj -> dt_proj`, selective `B/C`, state dynamics `A`, gate와 output projection까지 gradient가 연결되는지 확인한다.


In [ ]:
mamba.train()
mamba.zero_grad(
    set_to_none=True
)

output, _ = mamba(sequence)

loss = output.square().mean()
loss.backward()

parameters = dict(
    mamba.named_parameters()
)

for name in [
    "in_proj.weight",
    "conv1d.weight",
    "x_proj.weight",
    "dt_proj.weight",
    "dt_proj.bias",
    "A_log",
    "D",
    "out_proj.weight",
]:
    gradient = parameters[
        name
    ].grad

    print(
        name,
        gradient.norm().item(),
    )


## References and provenance

**State Space Models** — fixed-size recurrent state와 continuous-to-discrete transition을 먼저 분리해서 본다.

**Mamba: Linear-Time Sequence Modeling with Selective State Spaces** 및 공식 `state-spaces/mamba` 구현 — `in_proj`, causal depthwise convolution, `x_proj -> [dt_rank, B, C]`, `dt_proj`, learned diagonal `A`, `D`, selective scan, `SiLU(z)` gate, `out_proj` 구조를 반영했다.

공식 CUDA selective-scan / causal-conv fused kernel은 성능 구현이므로 이 최소구현에서는 sequential PyTorch reference scan으로 바꾼다. 그러나 학습 parameterization 자체는 유지한다.
